In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold.fhir;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.fhir.dim_patient (
    patient_key BIGINT GENERATED ALWAYS AS IDENTITY,
    patient_id STRING,
    active BOOLEAN,
    gender STRING,
    birth_date STRING,
    family_name STRING,
    given_name STRING,
    contact_1 STRING,
    contact_2 STRING,
    city STRING,
    state STRING,
    postal_code STRING,
    country STRING,
    marital_status STRING,
    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.fhir.dim_encounter (
    encounter_key BIGINT GENERATED ALWAYS AS IDENTITY,
    encounter_id STRING,
    organization_id STRING,
    status STRING,
    period_start STRING,
    period_end STRING
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.fhir.fact_encounter (
    encounter_key BIGINT,
    patient_key BIGINT,
    status STRING,
    period_start STRING,
    period_end STRING
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.fhir.fact_condition (
    condition_id STRING,
    patient_key BIGINT,
    encounter_key BIGINT,
    condition_text STRING,
    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.fhir.fact_observation (
    observation_id STRING,
    patient_key BIGINT,
    observation_text STRING,
    loinc_code STRING,
    category_code STRING,
    value_string STRING,
    value_quantity DOUBLE,
    value_unit STRING,
    effective_datetime STRING
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *

In [0]:
patient_df = spark.table("silver.fhir.patient")
encounter_df = spark.table("silver.fhir.encounter")
condition_df = spark.table("silver.fhir.condition")
observation_df = spark.table("silver.fhir.observation")

In [0]:


dim_patient_df = (
    patient_df
    .filter(col("is_current") == True)
    .select(
        "patient_id",
        "active",
        "gender",
        "birth_date",
        "family_name",
        "given_name",
        "contact_1",
        "contact_2",
        "city",
        "state",
        "postal_code",
        "country",
        col("marital_status"),
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
)

# ATOMIC overwrite (single commit)
(dim_patient_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("gold.fhir.dim_patient"))

In [0]:
dim_encounter_df = (
    encounter_df
    .filter(col("is_current") == True)
    .select(
        "encounter_id",
        "organization_id",
        "status",
        "period_start",
        "period_end"
    )
)

(dim_encounter_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("gold.fhir.dim_encounter"))

In [0]:
dim_patient = spark.table("gold.fhir.dim_patient")
dim_encounter = spark.table("gold.fhir.dim_encounter")

In [0]:
fact_encounter_df = (
    encounter_df.alias("s")
    .filter(col("s.is_current") == True)
    .join(dim_patient.alias("p"), col("s.patient_id") == col("p.patient_id"))
    .join(dim_encounter.alias("e"), col("s.encounter_id") == col("e.encounter_id"))
    .select(
        col("e.encounter_id"),
        col("p.patient_id"),
        col("s.status"),
        col("s.period_start"),
        col("s.period_end")
    )
)

(fact_encounter_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("gold.fhir.fact_encounter"))

In [0]:
fact_condition_df = (
    condition_df.alias("s")
    .filter(col("s.is_current") == True)
    .join(dim_patient.alias("p"), "patient_id")
    .join(dim_encounter.alias("e"), "encounter_id", "left")
    .select(
        col("s.condition_id"),
        col("p.patient_id"),
        col("e.encounter_id"),
        col("s.condition_text"),
        col("s.effective_start_date"),
        col("s.effective_end_date"),
        col("s.is_current")
    )
)

(fact_condition_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("gold.fhir.fact_condition"))

In [0]:

fact_observation_df = (
    observation_df.alias("s")
    .filter(col("s.is_current") == True)
    .join(dim_patient.alias("p"), col("s.patient_id") == col("p.patient_id"))
    .select(
        col("s.observation_id"),
        col("p.patient_id"),
        col("s.observation_text"),
        col("s.loinc_code"),
        col("s.category_code"),
        col("s.value_string"),
        col("s.value_quantity"),
        col("s.value_unit"),
        col("s.effective_datetime")
    )
)

(fact_observation_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("gold.fhir.fact_observation"))

In [0]:
%sql
select * from gold.fhir.fact_observation limit 10